# GEC inline-edit fine-tuning — DPO bonus stage

**Inputs:** the SFT adapter from notebook 01 and `data/processed/dpo.jsonl` (built by `scripts/build_dpo_pairs.py`).

**Output:** a DPO-tuned adapter pushed to `<user>/qwen2.5-3b-gec-dpo`.

Wall time ≈ 25–35 min on free Colab T4.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub

## 2. Clone the project repo

In [ ]:
import os
REPO_URL = os.environ.get('GEC_REPO_URL', 'https://github.com/LittleHydron/gec-inline')
!test -d gec-inline || git clone --depth 1 $REPO_URL
%cd gec-inline

## 3. Load the merged SFT model + a fresh LoRA
Notebook 01 pushed a merged 16-bit copy of base+SFT. We reload it in 4-bit and attach a **new** trainable LoRA, so DPO updates a fresh adapter while 'adapter disabled' (TRL's `ref_model=None` trick) is exactly the SFT policy — the frozen reference DPO needs.

(Loading the *adapter* repo here instead would fail: Unsloth returns a PeftModel and `get_peft_model` refuses to stack a second adapter.)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024
HF_USER = 'Lopato4ka'  # <- EDIT if you are not Lopato4ka
SFT_MERGED = f'{HF_USER}/qwen2.5-3b-gec-sft-merged'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SFT_MERGED,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha = 32,
    lora_dropout = 0.0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

## 4. Prepare DPO pairs
Each pair has a `prompt` (rendered text), a `chosen` (gold bracketed) and a `rejected` (corrupted / SFT-mistaken) completion. We re-wrap the prompt with the Qwen chat template so the model sees the same input format as during SFT.

In [ ]:
import json
from datasets import load_dataset
from gec.prompts import SYSTEM_PROMPT, build_user_message

def render(ex):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_message(ex['source'])},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {'prompt': prompt, 'chosen': ex['chosen'], 'rejected': ex['rejected']}

raw = load_dataset('json', data_files='data/processed/dpo.jsonl', split='train')
ds = raw.map(render, remove_columns=raw.column_names)
print('rows:', len(ds))
print(ds[0])

## 5. Configure DPO

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

USE_BF16 = is_bfloat16_supported()

config = DPOConfig(
    output_dir = 'outputs/dpo',
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    warmup_ratio = 0.05,
    num_train_epochs = 1,
    learning_rate = 5e-6,
    lr_scheduler_type = 'cosine',
    weight_decay = 0.0,
    optim = 'adamw_8bit',
    logging_steps = 20,
    save_strategy = 'epoch',
    save_total_limit = 1,
    seed = 3407,
    bf16 = USE_BF16,
    fp16 = not USE_BF16,
    beta = 0.1,
    max_length = MAX_SEQ_LEN,
    max_prompt_length = 512,
    report_to = 'none',
)

trainer = DPOTrainer(
    model = model,
    ref_model = None,  # TRL builds the ref from the model w/ adapter disabled
    args = config,
    train_dataset = ds,
    tokenizer = tokenizer,
)

## 6. Train

In [ ]:
stats = trainer.train()
stats.metrics

## 7. Push the DPO adapter to the Hub

In [ ]:
from huggingface_hub import login
login()

DPO_REPO = f'{HF_USER}/qwen2.5-3b-gec-dpo'

model.push_to_hub(DPO_REPO, private=False)
tokenizer.push_to_hub(DPO_REPO, private=False)
print('pushed:', DPO_REPO)

## 8. Generate eval predictions

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
from unsloth import FastLanguageModel
from gec.inference import generate_batch

FastLanguageModel.for_inference(model)

EVAL_PATH = 'data/processed/eval_bea_dev.jsonl'
OUT_PATH  = 'results/predictions/dpo_bea_dev.jsonl'
Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)

rows = [json.loads(line) for line in open(EVAL_PATH)]
sentences = [r['source'] for r in rows]
results = []
for start in tqdm(range(0, len(sentences), 8)):
    batch = sentences[start:start+8]
    for r in generate_batch(batch, tokenizer, model, batch_size=8):
        results.append({'source': r.source, 'raw': r.raw,
                        'corrected': r.corrected, 'parse_ok': r.parse_ok})

with open(OUT_PATH, 'w') as f:
    for r in results:
        f.write(json.dumps(r) + '\n')
print('wrote', OUT_PATH, len(results))